# Part 1: ROUGE and BLEU only

Run the code cells in order with **Shift+Enter**. When prompted, upload the two supplied JSONL files. No data-folder path is required.

The goal is to get the strongest scores from **ROUGE and BLEU**, with settings chosen on training biographies and accuracy measured on different biographies. There are no other metric families or combined models in this notebook.


## 1. Setup
Requires NumPy and NLTK. If missing, run `%pip install numpy==2.0.2 nltk==3.9.1 ipywidgets` once, then rerun this cell.

In [ ]:
from pathlib import Path
from collections import Counter
from functools import lru_cache
import csv
import html
import json
import math
import random
import re
import sys
import unicodedata
import numpy as np

try:
    from nltk.stem import PorterStemmer
except ImportError:
    # Reuse the packages already installed in the original project.
    for folder in [Path.cwd(), *Path.cwd().parents]:
        sys.path.append(str(folder / "Part1" / "_packages"))
    from nltk.stem import PorterStemmer

SEED = 42
print("Setup ready.")

## 2. Upload data
Run the next cell and select both JSONL files. In Jupyter, wait for the upload to finish before running the following cell. In Colab, use the file picker that opens.

In [ ]:
# Run this cell, then select both supplied JSONL files.
uploaded_files = {}
uploader = None

try:
    from google.colab import files
except ImportError:
    try:
        import ipywidgets as widgets
        from IPython.display import display
    except ImportError as error:
        raise ImportError("Run %pip install ipywidgets, then rerun this cell.") from error
    uploader = widgets.FileUpload(accept=".jsonl", multiple=True)
    print("Select both JSONL files below. Wait for the upload to finish, then run the next cell.")
    display(uploader)
else:
    print("Select dev_labeled_ChatGPT.jsonl and passages_bm25_ChatGPT_humfacts.jsonl.")
    uploaded_files = files.upload()


### Read the uploaded files
Run after both uploads finish. `IR` is mapped to `NS`.

In [ ]:
# Read the uploaded files directly from memory; no data path is assumed.
if uploader is not None:
    values = uploader.value
    if isinstance(values, dict):  # ipywidgets 7
        uploaded_files = {name: bytes(item["content"]) for name, item in values.items()}
    else:  # ipywidgets 8
        uploaded_files = {item["name"]: bytes(item["content"]) for item in values}

required_files = ["dev_labeled_ChatGPT.jsonl", "passages_bm25_ChatGPT_humfacts.jsonl"]
missing = [name for name in required_files if name not in uploaded_files]
if missing:
    raise ValueError("Upload both files in the previous cell, then rerun this cell. Missing: " + ", ".join(missing))


def read_uploaded_jsonl(name):
    text = bytes(uploaded_files[name]).decode("utf-8-sig")
    return [json.loads(line) for line in text.splitlines() if line.strip()]


retrieval = read_uploaded_jsonl("passages_bm25_ChatGPT_humfacts.jsonl")
passage_lookup = {record["sent"]: record["passages"] for record in retrieval}
examples = []
for biography in read_uploaded_jsonl("dev_labeled_ChatGPT.jsonl"):
    for annotation in biography.get("annotations") or []:
        for fact in annotation.get("human-atomic-facts") or []:
            if fact["text"] in passage_lookup:
                examples.append({
                    "fact": fact["text"],
                    "passages": passage_lookup[fact["text"]],
                    "topic": biography["topic"],
                    "label": "NS" if fact["label"] == "IR" else fact["label"],
                })

if not examples:
    raise ValueError("The uploaded files contain no matching annotated facts.")
y = np.array([example["label"] == "S" for example in examples])
all_indices = np.arange(len(examples))
print(f"Loaded {len(examples)} facts about {len({e['topic'] for e in examples})} biographies.")
print("Labels:", dict(Counter(e["label"] for e in examples)))


## 3. Preprocessing
Compare stemming, stop-word removal, title-word removal and four evidence lengths: 8 tokens, fact length, twice fact length, or sentences. Numbers and negations remain. There are 32 settings; ROUGE and BLEU each choose their own best setting using training data.

In [ ]:
STOP = frozenset("""a an the is are was were be been being am to of in on at by for from
with as and or but that this these those it its his her their he she they who which
has have had do does did""".split())
porter = PorterStemmer()

@lru_cache(maxsize=50000)
def tokenize(text, stemming, remove_stopwords):
    cleaned = unicodedata.normalize("NFKC", html.unescape(re.sub(r"<[^>]+>", " ", text))).casefold()
    words = re.findall(r"[^\W_]+", cleaned)
    return tuple(porter.stem(w) if stemming else w for w in words
                 if not remove_stopwords or w not in STOP)

CONFIGS = [
    {"stemming": stem, "remove_stopwords": stop, "remove_title": title, "evidence": unit}
    for stem in (False, True)
    for stop in (False, True)
    for title in (False, True)
    for unit in ("window8", "fact_length", "twice_fact_length", "sentence")
]


def prepare(example, config):
    def tokens(text):
        return tokenize(text, config["stemming"], config["remove_stopwords"])
    title = set(tokens(example["passages"][0]["title"])) if example["passages"] and config["remove_title"] else set()
    def clean(text):
        return tuple(w for w in tokens(text) if w not in title)
    fact = clean(example["fact"])
    windows = []
    for passage in example["passages"]:
        if config["evidence"] == "sentence":
            windows.extend(clean(s) for s in re.split(r"</?s>|(?<=[.!?])\s+", passage["text"]) if s.strip())
        else:
            words = clean(passage["text"])
            length = 8 if config["evidence"] == "window8" else max(1, len(fact))
            if config["evidence"] == "twice_fact_length":
                length *= 2
            windows.extend(words[i:i + length] for i in range(0, len(words), max(1, length // 2)))
    return fact, list(dict.fromkeys(w for w in windows if w))

## 4. ROUGE and BLEU
ROUGE compares unigram/bigram overlap or the longest common subsequence, using recall or F1. BLEU compares orders 1, 2 and 4, both text directions, and two smoothing methods. BLEU-1 uses unigrams; BLEU-2/4 include word order. For short candidates, use only the available orders. Each fact takes its highest evidence-window score.

In [ ]:
def ngrams(ts, n):
    return Counter((tuple(ts[i:i + n]) for i in range(len(ts) - n + 1)))

def lcs(a, b):
    if len(a) > len(b):
        a, b = (b, a)
    previous = [0] * (len(a) + 1)
    for y in b:
        current = [0]
        for i, x in enumerate(a, 1):
            current.append(previous[i - 1] + 1 if x == y else max(previous[i], current[-1]))
        previous = current
    return previous[-1]

FAMILIES = {
    "ROUGE": ["ROUGE-1 recall", "ROUGE-1 F1", "ROUGE-2 recall", "ROUGE-2 F1", "ROUGE-L recall", "ROUGE-L F1"],
    "BLEU": ["BLEU-1", "BLEU-1 reverse"] + [
        f"BLEU-{order} {smoothing}{direction}"
        for order in (2, 4)
        for smoothing in ("add-one", "epsilon")
        for direction in ("", " reverse")
    ],
}
METRICS = [metric for variants in FAMILIES.values() for metric in variants]


def f1(precision, recall):
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def pair_scores(fact, evidence):
    values = dict.fromkeys(METRICS, 0.0)
    if not fact or not evidence:
        return values
    fact_grams = {n: ngrams(fact, n) for n in range(1, 5)}
    evidence_grams = {n: ngrams(evidence, n) for n in range(1, 5)}
    matched = {n: sum((fact_grams[n] & evidence_grams[n]).values()) for n in range(1, 5)}
    for n in (1, 2):
        reference_count = sum(fact_grams[n].values())
        candidate_count = sum(evidence_grams[n].values())
        recall = matched[n] / reference_count if reference_count else 0.0
        precision = matched[n] / candidate_count if candidate_count else 0.0
        values[f"ROUGE-{n} recall"] = recall
        values[f"ROUGE-{n} F1"] = f1(precision, recall)
    common = lcs(fact, evidence)
    recall, precision = common / len(fact), common / len(evidence)
    values["ROUGE-L recall"] = recall
    values["ROUGE-L F1"] = f1(precision, recall)

    for reverse in (False, True):
        candidate, reference = (evidence, fact) if reverse else (fact, evidence)
        suffix = " reverse" if reverse else ""
        if matched[1] == 0:
            continue
        brevity_penalty = math.exp(min(0.0, 1 - len(reference) / len(candidate)))
        for order in (1, 2, 4):
            for smoothing in (("none",) if order == 1 else ("add-one", "epsilon")):
                precisions = []
                for n in range(1, min(order, len(candidate)) + 1):
                    total = len(candidate) - n + 1
                    hit = matched[n]
                    if n == 1:
                        value = hit / total
                    elif smoothing == "add-one":
                        value = (hit + 1) / (total + 1)
                    else:
                        value = hit / total if hit else 0.1 / total
                    precisions.append(value)
                score = brevity_penalty * math.exp(sum(math.log(p) for p in precisions) / len(precisions))
                name = f"BLEU-1{suffix}" if order == 1 else f"BLEU-{order} {smoothing}{suffix}"
                values[name] = score
    return values


def score_example(example, config):
    fact, windows = prepare(example, config)
    best = dict.fromkeys(METRICS, 0.0)
    for window in windows:
        for metric, value in pair_scores(fact, window).items():
            best[metric] = max(best[metric], value)
    return best

## 5. Calculate scores
Scores depend only on the fact and its evidence. This step may take around a minute.

In [ ]:
score_bank = []
for i, config in enumerate(CONFIGS, 1):
    rows = [score_example(example, config) for example in examples]
    score_bank.append({metric: np.array([row[metric] for row in rows]) for metric in METRICS})
    if i % 8 == 0:
        print(f"Scored {i}/{len(CONFIGS)} preprocessing settings.")

## 6. Select settings
Use three training-only biography splits to choose a metric variant, preprocessing and threshold. ROUGE and BLEU are optimized separately.

In [ ]:
def tune(scores, labels):
    pairs = sorted(zip(scores, labels))
    correct = sum((y == 'S' for y in labels))
    best = (correct, 0.0)
    i = 0
    while i < len(pairs):
        value = pairs[i][0]
        while i < len(pairs) and pairs[i][0] == value:
            correct += -1 if pairs[i][1] == 'S' else 1
            i += 1
        threshold = (value + pairs[i][0]) / 2 if i < len(pairs) else math.nextafter(value, math.inf)
        if correct > best[0]:
            best = (correct, threshold)
    return dict(threshold=best[1], accuracy=best[0] / len(labels), correct=best[0])

def grouped_folds(indices, k, seed):
    topics = sorted({examples[int(i)]['topic'] for i in indices})
    random.Random(seed).shuffle(topics)
    topic_sets = [set(topics[j::k]) for j in range(k)]
    for topics_in_test in topic_sets:
        test = np.array([i for i in indices if examples[int(i)]['topic'] in topics_in_test], dtype=int)
        train = np.array([i for i in indices if examples[int(i)]['topic'] not in topics_in_test], dtype=int)
        assert set((examples[int(i)]['topic'] for i in train)).isdisjoint((examples[int(i)]['topic'] for i in test))
        yield (train, test)


def fit_threshold(scores, indices):
    return tune(scores[indices].tolist(), ["S" if label else "NS" for label in y[indices]])


def select_model(family, indices, seed):
    folds = list(grouped_folds(indices, 3, seed))
    best = None
    for config_index, bank in enumerate(score_bank):
        for metric in FAMILIES[family]:
            correct, count = 0, 0
            for train, validation in folds:
                threshold = fit_threshold(bank[metric], train)["threshold"]
                correct += int(np.sum((bank[metric][validation] >= threshold) == y[validation]))
                count += len(validation)
            accuracy = correct / count
            if best is None or accuracy > best["inner_accuracy"]:
                best = {"config_index": config_index, "metric": metric, "inner_accuracy": accuracy}
    return best

## 7. Run validation
Use five outer biography groups. Each fact is predicted once without its biography being used for that round’s model selection.

In [ ]:
oof = {family: np.zeros(len(examples), dtype=bool) for family in FAMILIES}
fold_choices = []
for fold, (train, test) in enumerate(grouped_folds(all_indices, 5, SEED), 1):
    for family in FAMILIES:
        selected = select_model(family, train, SEED + fold)
        scores = score_bank[selected["config_index"]][selected["metric"]]
        threshold = fit_threshold(scores, train)["threshold"]
        oof[family][test] = scores[test] >= threshold
        fold_choices.append({"fold": fold, "family": family, "variant": selected["metric"],
                             "config": CONFIGS[selected["config_index"]], "threshold": threshold})
    print(f"Fold {fold}/5 complete.")

validation_accuracy = {family: float(np.mean(predictions == y)) for family, predictions in oof.items()}

## 8. Results
Validation is the main result. Development scores are higher-risk estimates because the same labels are used for tuning and evaluation. This previously explored dataset is not a fresh external test.

In [ ]:
development_best = {}
for family in FAMILIES:
    candidates = []
    for config_index, bank in enumerate(score_bank):
        for metric in FAMILIES[family]:
            result = fit_threshold(bank[metric], all_indices)
            candidates.append({"config": CONFIGS[config_index], "metric": metric, **result})
    development_best[family] = max(candidates, key=lambda candidate: candidate["accuracy"])

print(f"{'Method':<10} {'Validation':>14} {'Development (tuned)':>24}")
print("-" * 50)
for family in FAMILIES:
    print(f"{family:<10} {validation_accuracy[family]:>13.2%} {development_best[family]['accuracy']:>23.2%}")
print("\nValidation: biographies excluded from training.")
print("Development: same labels used for tuning and evaluation; optimistic.")


## 9. Fit the final classifiers
Refit settings selected by training-data cross-validation on all facts. Use `FactChecker("ROUGE")` or `FactChecker("BLEU")`, then call `.predict(fact, passages)`. The final selected variant need not be the one with the highest optimistic development score.

In [ ]:
final_settings = {}
for family in FAMILIES:
    selected = select_model(family, all_indices, SEED)
    scores = score_bank[selected["config_index"]][selected["metric"]]
    final_settings[family] = {
        "config": CONFIGS[selected["config_index"]],
        "metric": selected["metric"],
        "threshold": fit_threshold(scores, all_indices)["threshold"],
    }

for family, setting in final_settings.items():
    print(f"{family}: {setting['metric']}, threshold {setting['threshold']:.4f}")
    print("Preprocessing:", setting["config"])


class FactChecker:
    def __init__(self, family):
        self.setting = final_settings[family]

    def predict(self, fact, passages):
        scores = score_example({"fact": fact, "passages": passages}, self.setting["config"])
        return "S" if scores[self.setting["metric"]] >= self.setting["threshold"] else "NS"


example = examples[0]
print("\nExample:", example["fact"])
for family in FAMILIES:
    print(family, FactChecker(family).predict(example["fact"], example["passages"]))


## 10. Save results
Write the summary and validation predictions to `rouge_bleu_results/` in the current working directory. The summary includes fitted settings and selected variants for each fold.

In [ ]:
OUTPUT = Path.cwd() / "rouge_bleu_results"
OUTPUT.mkdir(exist_ok=True)
summary = {
    "validation": "5 outer biography-grouped folds; 3 inner folds",
    "validation_accuracy": validation_accuracy,
    "average_validation_accuracy": sum(validation_accuracy.values()) / 2,
    "final_settings": final_settings,
    "best_development_tuned": development_best,
    "fold_choices": fold_choices,
}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
with (OUTPUT / "predictions.csv").open("w", encoding="utf-8", newline="") as stream:
    writer = csv.writer(stream)
    writer.writerow(["fact", "gold", "ROUGE", "BLEU"])
    for i, example in enumerate(examples):
        writer.writerow([example["fact"], example["label"]]
                        + ["S" if oof[family][i] else "NS" for family in FAMILIES])
print("Saved summary.json and predictions.csv in", OUTPUT)